# 🥗 Health-Personalized Food Recommender System
## Complete End-to-End Data Science Project

> **Course Project** | Data Science & Recommender Systems  
> Covers all work packages: Scraping · Annotation · Data Quality · Embeddings · Recommender · Evaluation · Tuning · Frontend

---

### 📚 Notebook Structure

| # | Section | Work Package |
|---|---------|-------------|
| 0 | Setup & Installation | — |
| 1 | Data Collection (Scraping) | WP: Data Scraping |
| 2 | Data Annotation (Label Studio export) | WP: Data Annotation |
| 3 | Data Quality Checks | WP: Data Quality |
| 4 | Vector Embeddings | WP: Vector Embeddings |
| 5 | Exploratory Data Analysis | — |
| 6 | Content-Based Filtering | WP: Recommender System |
| 7 | Collaborative Filtering (Matrix Factorisation) | WP: Recommender System |
| 8 | Hybrid Recommender | WP: Recommender System |
| 9 | Health Constraint Filter | WP: Recommender System |
| 10 | Perturbation Analysis | WP: Perturbation Analysis |
| 11 | Performance Evaluation (Precision@k, Recall@k) | WP: Performance Evaluation |
| 12 | Hyperparameter Tuning (Optuna) | WP: Hyperparameter Tuning |
| 13 | Experiment Logging (Weights & Biases) | WP: Experiments Logging |
| 14 | Frontend Demo (Streamlit code) | WP: Frontend Application |
| 15 | Full Pipeline Summary | — |


---
## Section 0 — Setup & Installation

Install all required libraries. Run this cell first.


In [1]:
# ── Install all dependencies ──────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q \
    requests beautifulsoup4 selenium \
    pandas numpy scikit-learn matplotlib seaborn \
    scipy surprise optuna wandb \
    streamlit plotly tqdm ipywidgets

print('✅ All packages installed')


✅ All packages installed


In [2]:
# ── Core imports (used throughout the notebook) ───────────────────────────
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from scipy.sparse import csr_matrix
from collections import defaultdict
import json, os, re, time, random

# Plotting style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#F8F7F4',
    'axes.grid':        True,
    'grid.color':       '#E0DED8',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
})
COLORS = {
    'purple': '#534AB7', 'teal': '#0F6E56', 'amber': '#BA7517',
    'coral':  '#993C1D', 'blue': '#185FA5', 'green': '#3B6D11',
    'red':    '#A32D2D', 'gray': '#5F5E5A',
}
np.random.seed(42); random.seed(42)
print('✅ Imports complete')


✅ Imports complete


---
## Section 1 — Data Collection (Web Scraping)
**Work Package: Data Scraping**

We scrape recipe data from a website using **BeautifulSoup**. Since live scraping depends
on network access and website structure, we provide:
1. The **real scraper code** you adapt for your chosen site
2. A **synthetic dataset generator** that mimics real scraped data for the rest of the notebook

### How scraping works
A website is HTML — a tree of tags. We:
1. Download the HTML with `requests.get(url)`
2. Parse the tree with `BeautifulSoup`
3. Find the tag containing the value we want with CSS selectors
4. Extract, clean, and save the text value


In [3]:
# ── 1.1  Real scraper skeleton (adapt to your chosen website) ───────────────
import requests
from bs4 import BeautifulSoup

def scrape_recipe(url: str) -> dict | None:
    """
    Scrape one recipe page and return a dict of nutritional values.
    Adapt the CSS selectors to match your target website.
    """
    headers = {'User-Agent': 'Mozilla/5.0'}   # polite header
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        resp.raise_for_status()
    except Exception as e:
        print(f'  ❌ Failed to fetch {url}: {e}')
        return None

    soup = BeautifulSoup(resp.text, 'html.parser')

    def extract(selector, default=None):
        """Helper: find tag and return stripped text, or default if missing."""
        el = soup.select_one(selector)
        return el.get_text(strip=True) if el else default

    def to_int(text):
        """Helper: pull first number from a string like '367 kcal'."""
        if text is None: return None
        m = re.search(r'[\d.]+', text)
        return float(m.group()) if m else None

    # ── Adapt these selectors to your target site ─────────────────────────────
    # Example selectors for BBC Good Food  (change for other sites)
    record = {
        'name':       extract('h1.heading-1') or extract('h1'),
        'url':        url,
        'calories':   to_int(extract('[data-ingredient-unit="kcal"]') or
                             extract('.nutrition__item--kcal .nutrition__value')),
        'protein_g':  to_int(extract('.nutrition__item--protein .nutrition__value')),
        'carbs_g':    to_int(extract('.nutrition__item--carbs .nutrition__value')),
        'fat_g':      to_int(extract('.nutrition__item--fat .nutrition__value')),
        'fiber_g':    to_int(extract('.nutrition__item--fibre .nutrition__value')),
        'sodium_mg':  to_int(extract('.nutrition__item--salt .nutrition__value')),
        'sugar_g':    to_int(extract('.nutrition__item--sugar .nutrition__value')),
    }
    return record


def scrape_multiple(urls: list[str], delay: float = 1.5) -> pd.DataFrame:
    """
    Scrape a list of URLs with a polite delay between requests.
    Returns a DataFrame of all scraped recipes.
    """
    records = []
    for i, url in enumerate(urls):
        print(f'  Scraping {i+1}/{len(urls)}: {url[:60]}...')
        rec = scrape_recipe(url)
        if rec:
            records.append(rec)
        time.sleep(delay)   # be polite — don't hammer the server
    return pd.DataFrame(records)


# Example URLs — replace with your real target URLs
EXAMPLE_URLS = [
    'https://www.bbcgoodfood.com/recipes/grilled-salmon-with-roasted-tomatoes-capers',
    'https://www.bbcgoodfood.com/recipes/quinoa-superfood-salad',
]

# Uncomment to run live scraping:
# df_scraped = scrape_multiple(EXAMPLE_URLS)
# df_scraped.to_csv('data/recipes_scraped.csv', index=False)
print('✅ Scraper defined. Use scrape_multiple(urls) to run live.')


✅ Scraper defined. Use scrape_multiple(urls) to run live.


In [ ]:
# ── 1.2  Synthetic dataset (simulates real scraped data) ─────────────────────
# This gives us 120 recipes with realistic nutritional values and categories.
# In your real project, replace this with your scraped CSV.

RECIPE_TEMPLATES = [
    # (name_pattern, cal_range, prot_range, carb_range, fat_range, fiber_range, sodium_range, sugar_range, cuisine, meal_type)
    ('Grilled Salmon {i}',      (300,420),(35,45),(0,5),  (18,28),(0,2),  (150,300),(0,2),  'seafood',     'dinner'),
    ('Tuna Salad {i}',          (200,300),(25,35),(5,15), (10,18),(2,5),  (300,500),(2,5),  'seafood',     'lunch'),
    ('Quinoa Bowl {i}',         (350,420),(12,18),(55,65),(6,12), (6,10), (180,300),(3,6),  'vegetarian',  'lunch'),
    ('Lentil Soup {i}',         (280,360),(16,22),(40,55),(5,10), (10,16),(400,600),(4,8),  'vegetarian',  'dinner'),
    ('Chicken Breast {i}',      (280,380),(40,55),(0,8),  (8,18), (0,2),  (200,400),(0,2),  'poultry',     'dinner'),
    ('Greek Salad {i}',         (180,260),(6,12), (10,20),(12,20),(2,5),  (350,550),(5,10), 'mediterranean','lunch'),
    ('Oatmeal Breakfast {i}',   (280,380),(8,14), (50,65),(6,12), (5,9),  (100,200),(8,15), 'breakfast',   'breakfast'),
    ('Beef Steak {i}',          (420,560),(45,60),(0,5),  (25,40),(0,1),  (300,500),(0,2),  'meat',        'dinner'),
    ('Tiramisu {i}',            (380,500),(5,9),  (48,62),(18,26),(0,2),  (60,120), (28,42),'italian',     'dessert'),
    ('Cheesecake {i}',          (350,480),(6,10), (40,55),(20,32),(0,2),  (200,350),(28,40),'dessert',     'dessert'),
    ('Avocado Toast {i}',       (300,400),(10,16),(28,38),(16,24),(6,10), (300,480),(2,5),  'vegetarian',  'breakfast'),
    ('Veggie Stir Fry {i}',     (250,350),(8,14), (30,45),(10,18),(5,9),  (450,700),(6,12), 'asian',       'dinner'),
    ('Pasta Carbonara {i}',     (480,600),(18,28),(55,70),(20,32),(2,5),  (400,650),(3,6),  'italian',     'dinner'),
    ('Caesar Salad {i}',        (250,380),(12,20),(15,25),(16,26),(2,5),  (450,700),(3,7),  'salad',       'lunch'),
    ('Egg Omelette {i}',        (220,320),(18,26),(2,8),  (14,22),(0,2),  (250,450),(1,4),  'eggs',        'breakfast'),
]

records = []
for i_t, (pattern, cal, prot, carb, fat, fib, sod, sug, cuisine, meal) in enumerate(RECIPE_TEMPLATES):
    for i in range(1, 9):   # 8 variants per template = 120 recipes
        name = pattern.format(i=i)
        records.append({
            'recipe_id':   len(records) + 1,
            'name':        name,
            'cuisine':     cuisine,
            'meal_type':   meal,
            'calories':    round(np.random.uniform(*cal)),
            'protein_g':   round(np.random.uniform(*prot), 1),
            'carbs_g':     round(np.random.uniform(*carb), 1),
            'fat_g':       round(np.random.uniform(*fat), 1),
            'fiber_g':     round(np.random.uniform(*fib), 1),
            'sodium_mg':   round(np.random.uniform(*sod)),
            'sugar_g':     round(np.random.uniform(*sug), 1),
        })
        # inject some missing values to simulate real scraped data
        if random.random() < 0.08:  records[-1]['fiber_g']  = None
        if random.random() < 0.05:  records[-1]['sodium_mg']= None
        if random.random() < 0.03:  records[-1]['protein_g']= None

df_raw = pd.DataFrame(records)
os.makedirs('data', exist_ok=True)
df_raw.to_csv('data/recipes_raw.csv', index=False)

print(f'✅ Dataset created: {len(df_raw)} recipes, {df_raw.shape[1]} columns')
print(f'   Cuisines: {df_raw.cuisine.unique()}')
df_raw.head(8)


---
## Section 2 — Data Annotation
**Work Package: Data Annotation**

In a real project you export annotations from **Label Studio** as a JSON/CSV file.
Here we simulate what that export looks like and apply the same parsing logic.

**How Label Studio works:**
1. You import your scraped CSV into Label Studio
2. For each recipe you see the nutritional values and click health labels
3. You export the annotations as JSON
4. This code parses that export and merges labels into the recipe dataset

**Health label rules we apply:**
| Label | Rule |
|-------|------|
| `diabetic_ok` | carbs ≤ 45g AND sugar ≤ 10g |
| `low_sodium` | sodium ≤ 400mg |
| `high_fiber` | fiber ≥ 5g |
| `high_protein` | protein ≥ 30g |
| `heart_healthy` | saturated fat proxy: fat ≤ 15g AND sodium ≤ 500mg |
| `low_calorie` | calories ≤ 300 |
| `high_sugar` | sugar > 20g |
| `vegetarian` | cuisine in vegetarian list |


In [ ]:
# ── 2.1  Simulate Label Studio JSON export ───────────────────────────────────
# In your real project: load with  pd.read_json('annotations_export.json')

def simulate_label_studio_export(df: pd.DataFrame) -> list[dict]:
    """Simulate what a Label Studio JSON export looks like."""
    annotations = []
    for _, row in df.iterrows():
        annotations.append({
            'id':   int(row['recipe_id']),
            'data': {'recipe_name': row['name']},
            'annotations': [{
                'result': [{
                    'from_name': 'health_labels',
                    'value': {'choices': []},  # filled by annotator
                }]
            }]
        })
    return annotations

ls_export = simulate_label_studio_export(df_raw)
print(f'Label Studio export format (first record):')
print(json.dumps(ls_export[0], indent=2))


In [ ]:
# ── 2.2  Apply annotation rules (auto-label from nutritional values) ──────────
# This mirrors what a human annotator would click in Label Studio.
# You can override individual labels if the auto-rule is wrong.

VEGETARIAN_CUISINES = {'vegetarian', 'salad', 'breakfast', 'mediterranean'}

def annotate_recipe(row: pd.Series) -> dict:
    """Apply health label rules to one recipe row."""
    carbs   = row['carbs_g']   or 0
    sugar   = row['sugar_g']   or 0
    sodium  = row['sodium_mg'] or 0
    fiber   = row['fiber_g']   or 0
    protein = row['protein_g'] or 0
    fat     = row['fat_g']     or 0
    cal     = row['calories']  or 0
    cuisine = row['cuisine']

    return {
        'recipe_id':    int(row['recipe_id']),
        'diabetic_ok':  int(carbs <= 45 and sugar <= 10),
        'low_sodium':   int(sodium <= 400),
        'high_fiber':   int(fiber >= 5),
        'high_protein': int(protein >= 30),
        'heart_healthy':int(fat <= 15 and sodium <= 500),
        'low_calorie':  int(cal <= 300),
        'high_sugar':   int(sugar > 20),
        'vegetarian':   int(cuisine in VEGETARIAN_CUISINES),
    }

annotation_rows = [annotate_recipe(row) for _, row in df_raw.iterrows()]
df_labels = pd.DataFrame(annotation_rows)

# Merge annotations into main dataset
df_annotated = df_raw.merge(df_labels, on='recipe_id')
df_annotated.to_csv('data/recipes_annotated.csv', index=False)

print(f'✅ Annotations applied: {len(df_annotated)} recipes')
print('\nLabel distribution:')
label_cols = ['diabetic_ok','low_sodium','high_fiber','high_protein',
               'heart_healthy','low_calorie','high_sugar','vegetarian']
label_counts = df_annotated[label_cols].sum().sort_values(ascending=False)
print(label_counts.to_string())


In [ ]:
# ── 2.3  Visualise label distribution ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
colors_bar = [COLORS['teal'] if v > 40 else COLORS['purple']
               if v > 20 else COLORS['amber'] for v in label_counts.values]
bars = ax.bar(label_counts.index, label_counts.values, color=colors_bar,
               edgecolor='white', linewidth=0.5)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set_title('Health Label Frequency Across All Recipes', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of recipes')
ax.set_xlabel('Health label')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('plots/label_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Label distribution plot saved')


---
## Section 3 — Data Quality
**Work Package: Data Quality**

We apply three data quality metrics:
1. **Completeness** — fraction of non-missing values per column
2. **Consistency** — do calories match the macronutrient formula?
3. **Range validity** — are values within physiologically plausible bounds?

### Metric 1: Completeness
$$\text{Completeness}(col) = \frac{\text{non-missing values}}{\text{total rows}}$$

### Metric 2: Consistency
$$\text{Expected kcal} = \text{protein} \times 4 + \text{carbs} \times 4 + \text{fat} \times 9$$
$$\text{Error} = |\text{actual kcal} - \text{expected kcal}|$$

### Metric 3: Range Validity
Check that each feature falls within a physiologically plausible range.


In [ ]:
os.makedirs('plots', exist_ok=True)

# ── 3.1  Completeness ─────────────────────────────────────────────────────────
NUMERIC_COLS = ['calories','protein_g','carbs_g','fat_g','fiber_g','sodium_mg','sugar_g']

def completeness_report(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    total = len(df)
    rows = []
    for col in cols:
        present = df[col].notna().sum()
        missing = total - present
        pct     = present / total
        status  = '✅ OK' if pct >= 0.95 else '⚠️  Review' if pct >= 0.80 else '❌ Poor'
        action  = 'Proceed' if pct >= 0.95 else 'Mean impute' if pct >= 0.80 else 'Re-scrape'
        rows.append({'Column': col, 'Present': present, 'Missing': missing,
                     'Completeness': f'{pct:.1%}', 'Status': status, 'Action': action})
    return pd.DataFrame(rows)

completeness_df = completeness_report(df_annotated, NUMERIC_COLS)
print('=== COMPLETENESS REPORT ===')
print(completeness_df.to_string(index=False))


In [ ]:
# ── 3.2  Consistency check ─────────────────────────────────────────────────────
# Expected kcal = protein*4 + carbs*4 + fat*9
# A tolerance of ±80 kcal is normal (fibre, alcohol, rounding differ by database)

CALORIE_TOLERANCE = 80  # kcal

def check_consistency(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['expected_kcal'] = (df['protein_g'].fillna(0) * 4 +
                           df['carbs_g'].fillna(0)   * 4 +
                           df['fat_g'].fillna(0)     * 9).round()
    df['kcal_error']    = (df['calories'] - df['expected_kcal']).abs()
    df['consistent']    = df['kcal_error'] <= CALORIE_TOLERANCE
    return df

df_checked = check_consistency(df_annotated)
n_inconsistent = (~df_checked['consistent']).sum()
print(f'Consistency check:')
print(f'  Total recipes:     {len(df_checked)}')
print(f'  Consistent:        {df_checked["consistent"].sum()}')
print(f'  Inconsistent:      {n_inconsistent}  ({n_inconsistent/len(df_checked):.1%})')
print(f'\nWorked example:')
row = df_checked.iloc[0]
print(f'  {row["name"]}')
print(f'  protein={row["protein_g"]}g  carbs={row["carbs_g"]}g  fat={row["fat_g"]}g')
print(f'  Expected kcal: {row["protein_g"]*4:.0f} + {row["carbs_g"]*4:.0f} + {row["fat_g"]*9:.0f} = {row["expected_kcal"]:.0f}')
print(f'  Actual kcal:   {row["calories"]}')
print(f'  Error:         {row["kcal_error"]:.0f} kcal  → {"✅ OK" if row["consistent"] else "❌ Flag"}')


In [ ]:
# ── 3.3  Range validity check ─────────────────────────────────────────────────
VALID_RANGES = {
    'calories':  (50,  2000),
    'protein_g': (0,   200),
    'carbs_g':   (0,   300),
    'fat_g':     (0,   150),
    'fiber_g':   (0,   60),
    'sodium_mg': (0,   4000),
    'sugar_g':   (0,   150),
}

print('=== RANGE VALIDITY REPORT ===')
for col, (lo, hi) in VALID_RANGES.items():
    vals = df_annotated[col].dropna()
    out_of_range = ((vals < lo) | (vals > hi)).sum()
    status = '✅' if out_of_range == 0 else f'⚠️  {out_of_range} rows out of range'
    print(f'  {col:<12}  range [{lo}, {hi}]   {status}')


In [ ]:
# ── 3.4  Fix missing values: mean imputation ─────────────────────────────────
def impute_missing(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """
    Replace NaN in numeric columns with the column mean.
    Shows a before/after report.
    """
    df = df.copy()
    print('=== MEAN IMPUTATION ===')
    for col in cols:
        n_missing = df[col].isna().sum()
        if n_missing > 0:
            col_mean = df[col].mean()
            df[col]  = df[col].fillna(col_mean)
            print(f'  {col:<12}  {n_missing} values imputed with mean = {col_mean:.2f}')
    print('\n✅ Imputation complete. Missing values remaining:')
    print(df[cols].isna().sum().to_string())
    return df

df_clean = impute_missing(df_annotated, NUMERIC_COLS)
df_clean.to_csv('data/recipes_clean.csv', index=False)
print(f'\n✅ Clean dataset saved: {len(df_clean)} recipes')


In [ ]:
# ── 3.5  Data quality visualisation ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot 1: Completeness bar chart
comp_vals = [df_annotated[c].notna().mean() for c in NUMERIC_COLS]
colors_comp = [COLORS['teal'] if v >= 0.95 else COLORS['amber']
                if v >= 0.80 else COLORS['red'] for v in comp_vals]
axes[0].barh(NUMERIC_COLS, comp_vals, color=colors_comp, edgecolor='white')
axes[0].axvline(0.95, color=COLORS['red'], linestyle='--', linewidth=1, label='95% threshold')
axes[0].set_xlim(0, 1.05)
axes[0].set_xlabel('Completeness')
axes[0].set_title('Column Completeness', fontweight='bold')
axes[0].legend()
for i, v in enumerate(comp_vals):
    axes[0].text(v + 0.005, i, f'{v:.1%}', va='center', fontsize=9)

# Plot 2: Calorie error distribution
axes[1].hist(df_checked['kcal_error'].dropna(), bins=30,
              color=COLORS['purple'], alpha=0.8, edgecolor='white')
axes[1].axvline(CALORIE_TOLERANCE, color=COLORS['red'], linestyle='--',
                linewidth=1.5, label=f'Tolerance = {CALORIE_TOLERANCE} kcal')
axes[1].set_xlabel('|Actual kcal − Expected kcal|')
axes[1].set_ylabel('Number of recipes')
axes[1].set_title('Calorie Consistency Error Distribution', fontweight='bold')
axes[1].legend()

plt.suptitle('Data Quality Report', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plots/data_quality.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 4 — Vector Embeddings
**Work Package: Vector Embeddings**

We convert every recipe into a numerical vector that the model can do maths on.

**Recipe vector** = [normalised nutritional features] + [binary health labels]

**Normalisation formula:**
$$x_{\text{norm}} = \frac{x}{x_{\text{max}}}$$

So a vector for Grilled Salmon might look like:
```
[ 0.18,  0.27,  0.00,  0.22,  0.00,  1, 1, 0, 1, 1, 0, 0, 1 ]
  cal   prot  carbs   fat  fiber  diab_ok  low_na  high_fib ...
```


In [ ]:
# ── 4.1  Build recipe feature matrix R ───────────────────────────────────────

# Feature maximums for normalisation
FEATURE_MAX = {
    'calories':  2000,
    'protein_g': 150,
    'carbs_g':   300,
    'fat_g':     100,
    'fiber_g':   50,
    'sodium_mg': 4000,
    'sugar_g':   150,
}

NUTRITION_COLS = list(FEATURE_MAX.keys())
LABEL_COLS     = ['diabetic_ok','low_sodium','high_fiber','high_protein',
                   'heart_healthy','low_calorie','high_sugar','vegetarian']

def build_recipe_vectors(df: pd.DataFrame) -> tuple[np.ndarray, list, list]:
    """
    Returns:
      R          - (N_recipes x N_features) float matrix
      feature_names - list of feature names
      recipe_ids    - list of recipe IDs in row order
    """
    # Normalise nutritional columns
    nutrition = df[NUTRITION_COLS].copy()
    for col, max_val in FEATURE_MAX.items():
        nutrition[col] = nutrition[col] / max_val

    # Concatenate with label columns
    labels      = df[LABEL_COLS].astype(float)
    feature_df  = pd.concat([nutrition, labels], axis=1)
    feature_names = list(feature_df.columns)

    R = feature_df.values
    return R, feature_names, list(df['recipe_id'])

R, FEATURE_NAMES, RECIPE_IDS = build_recipe_vectors(df_clean)

print(f'✅ Recipe matrix R shape: {R.shape}  ({R.shape[0]} recipes × {R.shape[1]} features)')
print(f'\nFeature names: {FEATURE_NAMES}')
print(f'\nExample — first recipe vector:')
print(f'  Name:  {df_clean.iloc[0]["name"]}')
for name, val in zip(FEATURE_NAMES, R[0]):
    print(f'  {name:<15} {val:.4f}')


In [ ]:
# ── 4.2  Build user profiles ──────────────────────────────────────────────────
# In a real system users fill a health questionnaire. Here we define 6 demo users.

def make_user_vector(target_cal, target_prot, max_carbs, max_fat, min_fiber,
                     diabetic, hypertensive, lactose_intol, vegetarian_pref) -> np.ndarray:
    """
    Build a user preference vector in the same feature space as recipe vectors.
    Nutritional targets are normalised with the same FEATURE_MAX values.
    Health condition flags are 1/0.
    """
    nutrition_part = np.array([
        target_cal  / FEATURE_MAX['calories'],
        target_prot / FEATURE_MAX['protein_g'],
        max_carbs   / FEATURE_MAX['carbs_g'],
        max_fat     / FEATURE_MAX['fat_g'],
        min_fiber   / FEATURE_MAX['fiber_g'],
        0,   # sodium — not a user goal directly
        0,   # sugar  — handled via diabetic flag
    ])
    label_part = np.array([
        float(diabetic),           # wants diabetic_ok recipes
        float(hypertensive),       # wants low_sodium recipes
        float(min_fiber >= 5),     # wants high_fiber recipes
        float(target_prot >= 30),  # wants high_protein recipes
        float(hypertensive),       # wants heart_healthy recipes
        float(target_cal <= 300),  # wants low_calorie recipes
        0.0,                       # does NOT want high_sugar
        float(vegetarian_pref),    # wants vegetarian recipes
    ])
    return np.concatenate([nutrition_part, label_part])

USERS = {
    'alice':  {'vec': make_user_vector(400, 40, 45,  25, 5,  True,  False, False, False),
                'profile': 'Diabetic, high-protein goal, low-carb'},
    'bob':    {'vec': make_user_vector(500, 30, 200, 40, 3,  False, False, False, False),
                'profile': 'Healthy, no restrictions'},
    'carol':  {'vec': make_user_vector(350, 15, 200, 20, 10, False, False, True,  True),
                'profile': 'Vegetarian, lactose intolerant, high-fiber'},
    'david':  {'vec': make_user_vector(600, 50, 50,  40, 2,  False, True,  False, False),
                'profile': 'Hypertensive, muscle-building goal'},
    'emma':   {'vec': make_user_vector(300, 20, 100, 15, 8,  False, False, False, True),
                'profile': 'Vegetarian, weight-loss goal'},
    'frank':  {'vec': make_user_vector(700, 55, 100, 50, 2,  False, False, False, False),
                'profile': 'Athlete, high-calorie high-protein'},
}

print('✅ User profiles created:')
for name, info in USERS.items():
    print(f'  {name:<8}  {info["profile"]}')


In [ ]:
# ── 4.3  Visualise embedding space (PCA to 2D) ───────────────────────────────
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
R_2d = pca.fit_transform(R)

fig, ax = plt.subplots(figsize=(11, 7))

cuisine_palette = {
    'seafood':       COLORS['teal'],
    'vegetarian':    COLORS['green'],
    'poultry':       COLORS['blue'],
    'meat':          COLORS['coral'],
    'italian':       COLORS['red'],
    'dessert':       COLORS['amber'],
    'asian':         COLORS['purple'],
    'salad':         '#5AA065',
    'breakfast':     '#A06C2C',
    'eggs':          '#8E7A1B',
    'mediterranean': '#2E7DA6',
}

for cuisine, color in cuisine_palette.items():
    mask = df_clean['cuisine'] == cuisine
    ax.scatter(R_2d[mask, 0], R_2d[mask, 1], c=color, alpha=0.7,
                s=60, label=cuisine, edgecolors='white', linewidth=0.4)

# Plot user vectors (projected into the same space)
U_2d = pca.transform(np.array([u['vec'] for u in USERS.values()]))
for i, (uname, info) in enumerate(USERS.items()):
    ax.scatter(U_2d[i, 0], U_2d[i, 1], c='black', s=160,
                marker='*', zorder=5)
    ax.annotate(uname, (U_2d[i, 0], U_2d[i, 1]),
                xytext=(6, 6), textcoords='offset points',
                fontsize=9, fontweight='bold', color='black')

ax.set_title('Recipe Embedding Space (PCA 2D projection)\n'
              'Stars = user vectors, dots = recipes coloured by cuisine',
              fontsize=12, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
ax.legend(loc='upper right', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('plots/embedding_space.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Embedding space plot saved')


---
## Section 5 — Exploratory Data Analysis

Before modelling, understand the data distributions and feature correlations.


In [ ]:
# ── 5.1  Nutrition distribution per cuisine ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

plot_features = ['calories','protein_g','carbs_g','fat_g','fiber_g','sodium_mg']
for i, feat in enumerate(plot_features):
    ax = axes[i]
    # boxplot by cuisine
    grouped = [df_clean[df_clean['cuisine']==c][feat].dropna().values
                for c in cuisine_palette.keys()]
    bp = ax.boxplot(grouped, patch_artist=True, vert=True,
                     medianprops=dict(color='black', linewidth=1.5))
    for patch, color in zip(bp['boxes'], cuisine_palette.values()):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_xticks(range(1, len(cuisine_palette)+1))
    ax.set_xticklabels(list(cuisine_palette.keys()), rotation=35, ha='right', fontsize=7)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_ylabel('g / mg / kcal')

plt.suptitle('Nutritional Distribution by Cuisine', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plots/eda_nutrition.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 5.2  Correlation heatmap ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
corr = df_clean[NUTRITION_COLS + LABEL_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
             center=0, ax=ax, linewidths=0.3, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 6 — Content-Based Filtering
**Work Package: Recommender System**

Content-based filtering recommends recipes that are **similar to the user's preference vector**.
We measure similarity using cosine similarity.

### Cosine Similarity Formula
$$\cos(\mathbf{u}, \mathbf{r}) = \frac{\mathbf{u} \cdot \mathbf{r}}{\|\mathbf{u}\| \times \|\mathbf{r}\|}$$

- Result = **1.0** → perfectly similar (same direction)
- Result = **0.0** → unrelated (90° angle)
- Result = **-1.0** → opposite

### Worked example (manual calculation):
```
Alice  u = [0.20, 0.27, 0.15, 0.25, 0.10,  1, 0, 0, 1, ...]
Salmon r = [0.18, 0.27, 0.00, 0.22, 0.00,  1, 1, 0, 1, ...]

dot product = (0.20×0.18) + (0.27×0.27) + ... = high
Tiramisu    = low dot product  (different nutrition, different labels)
```


In [ ]:
# ── 6.1  Compute cosine similarities (user vs all recipes) ───────────────────

def content_based_scores(user_vec: np.ndarray, R: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between user_vec and every recipe in R.
    Returns array of shape (N_recipes,) with scores in [-1, 1].
    """
    # sklearn's cosine_similarity expects 2D arrays
    u = user_vec.reshape(1, -1)
    scores = cosine_similarity(u, R).flatten()
    return scores


def content_recommend(user_name: str, k: int = 10) -> pd.DataFrame:
    """
    Return top-k recipe recommendations for a user using content-based filtering.
    """
    user_vec = USERS[user_name]['vec']
    scores   = content_based_scores(user_vec, R)

    results = df_clean[['recipe_id','name','cuisine','meal_type',
                          'calories','protein_g','carbs_g']].copy()
    results['cb_score'] = scores
    results = results.sort_values('cb_score', ascending=False).head(k)
    return results.reset_index(drop=True)


# Demo: recommend for Alice
print(f'=== Content-Based Top-10 for Alice ({USERS["alice"]["profile"]}) ===')
alice_cb = content_recommend('alice', k=10)
print(alice_cb[['name','cuisine','calories','protein_g','carbs_g','cb_score']].to_string(index=False))


In [ ]:
# ── 6.2  Score matrix for all users ──────────────────────────────────────────
# Build full S[user, recipe] score matrix
S_content = np.array([content_based_scores(info['vec'], R)
                        for info in USERS.values()])

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(S_content, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_yticks(range(len(USERS)))
ax.set_yticklabels([f'{n}  ({USERS[n]["profile"][:30]})' for n in USERS])
ax.set_xlabel('Recipe index')
ax.set_title('Content-Based Score Matrix  S[user, recipe]', fontweight='bold')
plt.colorbar(im, ax=ax, label='Cosine similarity')
plt.tight_layout()
plt.savefig('plots/score_matrix_content.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Score matrix shape: {S_content.shape}  ({len(USERS)} users × {len(df_clean)} recipes)')


---
## Section 7 — Collaborative Filtering (Matrix Factorisation)
**Work Package: Recommender System**

Collaborative filtering learns from **user ratings** — not just recipe features.
The intuition: if Alice and Bob rate the same recipes similarly, recommend to Alice
what Bob liked that Alice hasn't tried yet.

### Matrix Factorisation
We decompose the sparse rating matrix **R** into two matrices:

$$R \approx U \times V^T$$

- **R** shape = (n_users × n_recipes) — sparse, with many ? (unknown) values
- **U** shape = (n_users × k) — user latent factors
- **V** shape = (n_recipes × k) — recipe latent factors
- **k** = number of hidden taste dimensions (hyperparameter)

### Training objective (minimise):
$$\mathcal{L} = \sum_{(u,r) \text{ known}} (R_{ur} - \mathbf{u}_u \cdot \mathbf{v}_r)^2 + \lambda(\|U\|^2 + \|V\|^2)$$


In [ ]:
# ── 7.1  Simulate user interaction data (ratings) ────────────────────────────
# In a real system, users rate meals via the frontend.
# We simulate: each user has rated ~15% of recipes (realistic sparsity).

N_SIMULATED_USERS = 80   # total simulated users
RATING_PROB       = 0.12  # probability a user rated any given recipe

random.seed(42)
rating_rows = []

# Named users get ratings consistent with their profiles
profile_cuisine_pref = {
    'alice': ['seafood'],
    'bob':   ['italian','dessert','meat'],
    'carol': ['vegetarian','salad','mediterranean'],
    'david': ['meat','poultry','seafood'],
    'emma':  ['vegetarian','salad'],
    'frank': ['meat','poultry'],
}

user_id_map = {name: i for i, name in enumerate(USERS.keys())}

for uid, (uname, info) in enumerate(USERS.items()):
    pref_cuisines = profile_cuisine_pref[uname]
    for rid, row in df_clean.iterrows():
        if random.random() < 0.20:   # 20% chance they rated it
            # preferred cuisines get higher ratings
            base = 4.0 if row['cuisine'] in pref_cuisines else 2.5
            rating = max(1, min(5, round(base + random.gauss(0, 0.8))))
            rating_rows.append({'user_id': uid, 'user_name': uname,
                                  'recipe_id': int(row['recipe_id']),
                                  'recipe_name': row['name'],
                                  'rating': rating})

# Add anonymous users
for uid in range(len(USERS), N_SIMULATED_USERS):
    pref = random.choice(list(cuisine_palette.keys()))
    for rid, row in df_clean.iterrows():
        if random.random() < RATING_PROB:
            base   = 4.0 if row['cuisine'] == pref else 2.8
            rating = max(1, min(5, round(base + random.gauss(0, 0.9))))
            rating_rows.append({'user_id': uid, 'user_name': f'user_{uid}',
                                  'recipe_id': int(row['recipe_id']),
                                  'recipe_name': row['name'],
                                  'rating': rating})

df_ratings = pd.DataFrame(rating_rows)
df_ratings.to_csv('data/ratings.csv', index=False)

n_users   = df_ratings['user_id'].nunique()
n_recipes = df_ratings['recipe_id'].nunique()
density   = len(df_ratings) / (n_users * n_recipes)
print(f'✅ Ratings dataset:')
print(f'   Total ratings: {len(df_ratings)}')
print(f'   Users:         {n_users}')
print(f'   Recipes rated: {n_recipes}')
print(f'   Matrix density:{density:.2%}  (rest are unknown ?)')
print(f'   Rating distribution:')
print(df_ratings['rating'].value_counts().sort_index().to_string())


In [ ]:
# ── 7.2  Matrix factorisation with Surprise SVD ──────────────────────────────
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split as surprise_split

# Load data into Surprise format
reader  = Reader(rating_scale=(1, 5))
data_sv = Dataset.load_from_df(df_ratings[['user_id','recipe_id','rating']], reader)

trainset_sv, testset_sv = surprise_split(data_sv, test_size=0.2, random_state=42)

# Train SVD (matrix factorisation)
svd = SVD(n_factors=50, n_epochs=30, lr_all=0.005, reg_all=0.1, random_state=42)
svd.fit(trainset_sv)

# Evaluate on test set
predictions = svd.test(testset_sv)
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions,  verbose=False)
print(f'\n✅ SVD trained')
print(f'   RMSE on test set: {rmse:.4f}')
print(f'   MAE  on test set: {mae:.4f}')

# Show example predictions
print('\nExample predictions vs actual:')
print(f'{"User":<10} {"Recipe":<30} {"Actual":>8} {"Predicted":>10}')
for pred in predictions[:8]:
    rname = df_clean[df_clean['recipe_id']==pred.iid]['name'].values
    rname = rname[0][:28] if len(rname) > 0 else str(pred.iid)
    print(f'{str(pred.uid):<10} {rname:<30} {pred.r_ui:>8.1f} {pred.est:>10.2f}')


In [ ]:
# ── 7.3  Manual matrix factorisation (from scratch for understanding) ─────────
# This shows the maths without a library, on a tiny 5x6 matrix.

print('=== Matrix Factorisation from scratch (5 users × 6 recipes, k=2) ===')
print()

# Tiny rating matrix  (0 = unknown)
R_mini = np.array([
    [5, 1, 0, 4, 0, 3],  # Alice
    [4, 0, 3, 0, 2, 0],  # Bob
    [0, 2, 5, 0, 5, 4],  # Carol
    [3, 0, 0, 5, 0, 4],  # David
    [0, 4, 4, 0, 3, 0],  # Emma
], dtype=float)

RECIPE_MINI = ['Salmon','Tiramisu','Quinoa','Steak','Salad','Pasta']
USER_MINI   = ['Alice','Bob','Carol','David','Emma']

n_users_m, n_items_m = R_mini.shape
k = 2          # latent factors
lr = 0.01      # learning rate
lam = 0.1      # regularisation
n_epochs = 200 # training iterations

# Initialise U and V randomly
np.random.seed(42)
U_mini = np.random.normal(0, 0.1, (n_users_m, k))
V_mini = np.random.normal(0, 0.1, (n_items_m, k))

# Stochastic Gradient Descent
losses = []
for epoch in range(n_epochs):
    epoch_loss = 0
    for u in range(n_users_m):
        for r in range(n_items_m):
            if R_mini[u, r] > 0:    # only update on known ratings
                err = R_mini[u, r] - np.dot(U_mini[u], V_mini[r])
                # gradient descent update
                U_mini[u] += lr * (err * V_mini[r] - lam * U_mini[u])
                V_mini[r] += lr * (err * U_mini[u] - lam * V_mini[r])
                epoch_loss += err**2
    losses.append(epoch_loss)

# Reconstruct full matrix
R_predicted = U_mini @ V_mini.T

print('Original sparse matrix (0 = unknown):')
print(pd.DataFrame(R_mini, index=USER_MINI, columns=RECIPE_MINI).to_string())
print('\nPredicted full matrix (filled in unknowns):')
df_pred = pd.DataFrame(R_predicted.round(2), index=USER_MINI, columns=RECIPE_MINI)
print(df_pred.to_string())
print('\nKey predictions:')
print(f'  Alice → Quinoa (was ?):  predicted = {R_predicted[0,2]:.2f} (actual task: will she like it?)')
print(f'  Carol → Salmon (was ?):  predicted = {R_predicted[2,0]:.2f} (Carol is vegetarian → low score makes sense)')
print(f'  Bob   → Steak  (was ?):  predicted = {R_predicted[1,3]:.2f}')


In [ ]:
# ── 7.4  Plot training loss curve ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses, color=COLORS['purple'], linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Training Loss (sum of squared errors)')
ax.set_title('Matrix Factorisation — Training Loss Convergence', fontweight='bold')
ax.annotate(f'Final loss: {losses[-1]:.4f}',
             xy=(len(losses)-1, losses[-1]),
             xytext=(-80, 20), textcoords='offset points',
             arrowprops=dict(arrowstyle='->', color=COLORS['amber']),
             color=COLORS['amber'], fontsize=9)
plt.tight_layout()
plt.savefig('plots/training_loss.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 7.5  Collaborative filtering recommendations ──────────────────────────────
def collab_recommend(user_id: int, k: int = 10,
                      already_rated: set = None) -> pd.DataFrame:
    """
    Recommend top-k recipes for a user using the trained SVD model.
    Excludes recipes the user has already rated.
    """
    all_recipe_ids = df_clean['recipe_id'].tolist()
    if already_rated is None:
        already_rated = set(df_ratings[df_ratings['user_id']==user_id]['recipe_id'])

    unseen = [r for r in all_recipe_ids if r not in already_rated]

    preds  = [(r, svd.predict(user_id, r).est) for r in unseen]
    preds  = sorted(preds, key=lambda x: x[1], reverse=True)[:k]

    rec_ids   = [p[0] for p in preds]
    rec_scores= [p[1] for p in preds]
    results   = df_clean[df_clean['recipe_id'].isin(rec_ids)].copy()
    score_map = dict(zip(rec_ids, rec_scores))
    results['cf_score'] = results['recipe_id'].map(score_map)
    return results.sort_values('cf_score', ascending=False).reset_index(drop=True)


print(f'=== Collaborative Filtering Top-10 for Alice (user_id=0) ===')
alice_rated = set(df_ratings[df_ratings['user_id']==0]['recipe_id'])
alice_cf = collab_recommend(0, k=10, already_rated=alice_rated)
print(alice_cf[['name','cuisine','calories','protein_g','carbs_g','cf_score']].to_string(index=False))


---
## Section 8 — Hybrid Recommender
**Work Package: Recommender System**

We blend both signals with a weight **alpha**:

$$\text{score}(u, r) = \alpha \times \cos(\mathbf{u}, \mathbf{r}) + (1-\alpha) \times \hat{R}_{ur}$$

- **alpha = 1.0** → pure content-based (works for new users with no ratings — cold start)
- **alpha = 0.0** → pure collaborative (requires rating history)
- **alpha = 0.6** → typical good balance

Alpha itself is a hyperparameter tuned in Section 12.


In [ ]:
# ── 8.1  Hybrid scoring function ─────────────────────────────────────────────
def hybrid_recommend(user_name: str, user_id: int,
                      alpha: float = 0.6, k: int = 10) -> pd.DataFrame:
    """
    Hybrid recommender combining content-based and collaborative scores.

    Parameters
    ----------
    user_name : name key in USERS dict
    user_id   : integer ID in the rating matrix
    alpha     : weight for content-based score (1-alpha for collaborative)
    k         : number of recommendations
    """
    user_vec = USERS[user_name]['vec']

    # Content-based scores for all recipes
    cb_scores = content_based_scores(user_vec, R)
    cb_scores = (cb_scores - cb_scores.min()) / (cb_scores.max() - cb_scores.min() + 1e-9)

    # Collaborative scores for all recipes
    cf_raw = np.array([svd.predict(user_id, rid).est for rid in df_clean['recipe_id']])
    cf_scores = (cf_raw - cf_raw.min()) / (cf_raw.max() - cf_raw.min() + 1e-9)

    # Blend
    final_scores = alpha * cb_scores + (1 - alpha) * cf_scores

    results = df_clean[['recipe_id','name','cuisine','meal_type',
                          'calories','protein_g','carbs_g','fat_g',
                          'sodium_mg','sugar_g'] + LABEL_COLS].copy()
    results['cb_score']    = cb_scores
    results['cf_score']    = cf_scores
    results['hybrid_score']= final_scores
    return results.sort_values('hybrid_score', ascending=False).head(k).reset_index(drop=True)


print('=== Hybrid Recommendations for each user (alpha=0.6, k=10) ===')
for uname, uid in [('alice',0),('carol',2),('david',3)]:
    recs = hybrid_recommend(uname, uid, alpha=0.6, k=5)
    print(f'\n{uname.upper()} ({USERS[uname]["profile"]}):')
    print(recs[['name','cuisine','hybrid_score','cb_score','cf_score']].to_string(index=False))


---
## Section 9 — Health Constraint Filter
**Work Package: Recommender System**

After scoring, we apply hard health rules that **cannot be overridden by the model**.
A diabetic user should never see high-carb desserts, regardless of how highly the model scored them.

| Condition | Rule | Threshold |
|-----------|------|-----------|
| Diabetes | carbs_g per meal | ≤ 45g |
| Hypertension | sodium_mg per meal | ≤ 600mg |
| Vegetarian | vegetarian label | must be True |
| Lactose intolerance | cuisine not in dairy list | — |


In [ ]:
# ── 9.1  Health constraint filter ───────────────────────────────────────────
DAIRY_CUISINES = {'italian', 'dessert'}  # simplified proxy

USER_CONSTRAINTS = {
    'alice':  {'diabetic': True,  'hypertensive': False, 'vegetarian': False, 'lactose_intol': False},
    'bob':    {'diabetic': False, 'hypertensive': False, 'vegetarian': False, 'lactose_intol': False},
    'carol':  {'diabetic': False, 'hypertensive': False, 'vegetarian': True,  'lactose_intol': True},
    'david':  {'diabetic': False, 'hypertensive': True,  'vegetarian': False, 'lactose_intol': False},
    'emma':   {'diabetic': False, 'hypertensive': False, 'vegetarian': True,  'lactose_intol': False},
    'frank':  {'diabetic': False, 'hypertensive': False, 'vegetarian': False, 'lactose_intol': False},
}

def health_filter(df_recs: pd.DataFrame, constraints: dict) -> pd.DataFrame:
    """
    Apply health constraints to a ranked recommendation list.
    Adds a 'blocked_reason' column; filtered output only contains allowed recipes.
    """
    df = df_recs.copy()
    df['blocked_reason'] = ''

    if constraints.get('diabetic'):
        mask = df['carbs_g'] > 45
        df.loc[mask, 'blocked_reason'] += 'high carbs; '

    if constraints.get('hypertensive'):
        mask = df['sodium_mg'] > 600
        df.loc[mask, 'blocked_reason'] += 'high sodium; '

    if constraints.get('vegetarian'):
        mask = df['vegetarian'] == 0
        df.loc[mask, 'blocked_reason'] += 'not vegetarian; '

    if constraints.get('lactose_intol'):
        mask = df['cuisine'].isin(DAIRY_CUISINES)
        df.loc[mask, 'blocked_reason'] += 'contains dairy; '

    df['allowed'] = df['blocked_reason'] == ''
    return df


def full_recommend(user_name: str, user_id: int,
                    alpha: float = 0.6, k: int = 10) -> pd.DataFrame:
    """Full pipeline: hybrid scoring → health filter → top-k."""
    candidates = hybrid_recommend(user_name, user_id, alpha=alpha, k=50)
    filtered   = health_filter(candidates, USER_CONSTRAINTS[user_name])
    allowed    = filtered[filtered['allowed']].head(k).reset_index(drop=True)
    return allowed


print('=== Alice: before and after health filter ===')
before = hybrid_recommend('alice', 0, alpha=0.6, k=10)
before_filtered = health_filter(before, USER_CONSTRAINTS['alice'])
print(before_filtered[['name','carbs_g','hybrid_score','allowed','blocked_reason']].to_string(index=False))


In [ ]:
# ── 9.2  Visualise filtering effect ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: before filter
colors_bar = [COLORS['red'] if not a else COLORS['teal']
               for a in before_filtered['allowed']]
axes[0].barh(before_filtered['name'], before_filtered['hybrid_score'],
              color=colors_bar, edgecolor='white')
axes[0].set_title('BEFORE health filter\n(red = blocked)', fontweight='bold')
axes[0].set_xlabel('Hybrid score')
axes[0].invert_yaxis()

# Right: after filter
after = before_filtered[before_filtered['allowed']]
axes[1].barh(after['name'], after['hybrid_score'],
              color=COLORS['teal'], edgecolor='white')
axes[1].set_title('AFTER health filter\n(Alice is diabetic → carbs ≤ 45g)', fontweight='bold')
axes[1].set_xlabel('Hybrid score')
axes[1].invert_yaxis()

plt.suptitle('Health Constraint Filtering for Alice', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/health_filter.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 10 — Perturbation Analysis
**Work Package: Perturbation Analysis**

How **robust** is our recommender to small errors in the input?

**Experiment:**
1. Start with clean user profile → get top-k recommendations
2. Introduce a small perturbation (flip a flag, add noise to a number)
3. Get top-k recommendations again
4. Measure overlap with **Jaccard similarity**: $J = |A \cap B| / |A \cup B|$

A robust system has J close to 1.0 — small input changes don't change recommendations much.
A fragile system has J close to 0.0 — worth investigating and fixing.


In [ ]:
# ── 10.1  Perturbation tests ─────────────────────────────────────────────────

def jaccard(set_a: set, set_b: set) -> float:
    if not set_a and not set_b: return 1.0
    return len(set_a & set_b) / len(set_a | set_b)


def perturb_user_vec(vec: np.ndarray, noise_std: float = 0.05) -> np.ndarray:
    """Add Gaussian noise to nutritional part of user vector."""
    perturbed = vec.copy()
    perturbed[:7] += np.random.normal(0, noise_std, 7)  # noise on nutrition part
    perturbed = np.clip(perturbed, 0, 1)
    return perturbed


def perturb_flags(vec: np.ndarray, flip_idx: int) -> np.ndarray:
    """Flip one binary flag in the user vector."""
    perturbed = vec.copy()
    perturbed[7 + flip_idx] = 1 - perturbed[7 + flip_idx]  # flip flag
    return perturbed


def perturbation_analysis(user_name: str, user_id: int,
                            k: int = 10, n_trials: int = 30) -> pd.DataFrame:
    """Run perturbation experiments and report Jaccard stability."""
    original_vec = USERS[user_name]['vec'].copy()

    # Baseline recommendations (no perturbation)
    cb_base   = content_based_scores(original_vec, R)
    top_base  = set(df_clean['recipe_id'].values[np.argsort(cb_base)[::-1][:k]])

    results = []

    # Test 1: Numeric noise at increasing levels
    for noise_std in [0.01, 0.03, 0.05, 0.10, 0.20]:
        jaccards = []
        for _ in range(n_trials):
            p_vec    = perturb_user_vec(original_vec, noise_std)
            cb_p     = content_based_scores(p_vec, R)
            top_p    = set(df_clean['recipe_id'].values[np.argsort(cb_p)[::-1][:k]])
            jaccards.append(jaccard(top_base, top_p))
        results.append({'perturbation': f'noise σ={noise_std}',
                         'mean_jaccard': np.mean(jaccards),
                         'std_jaccard':  np.std(jaccards),
                         'min_jaccard':  np.min(jaccards)})

    # Test 2: Flag flips
    for flag_i, flag_name in enumerate(LABEL_COLS[:4]):
        p_vec  = perturb_flags(original_vec, flag_i)
        cb_p   = content_based_scores(p_vec, R)
        top_p  = set(df_clean['recipe_id'].values[np.argsort(cb_p)[::-1][:k]])
        j      = jaccard(top_base, top_p)
        results.append({'perturbation': f'flip flag: {flag_name}',
                         'mean_jaccard': j, 'std_jaccard': 0, 'min_jaccard': j})

    return pd.DataFrame(results)


USERS['alice']['vec'] = USERS['alice']['vec']  # reset
perturbation_df = perturbation_analysis('alice', 0, k=10, n_trials=50)
print('=== Perturbation Analysis for Alice ===')
print(perturbation_df.to_string(index=False))


In [ ]:
# ── 10.2  Visualise perturbation robustness ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
colors_p = [COLORS['teal'] if v >= 0.7 else COLORS['amber']
              if v >= 0.4 else COLORS['red']
              for v in perturbation_df['mean_jaccard']]
bars = ax.bar(perturbation_df['perturbation'], perturbation_df['mean_jaccard'],
               color=colors_p, edgecolor='white', yerr=perturbation_df['std_jaccard'],
               capsize=4, error_kw={'color': COLORS['gray'], 'linewidth': 1})
ax.axhline(0.7, color=COLORS['green'], linestyle='--', linewidth=1.2, label='Robust threshold (0.7)')
ax.axhline(0.4, color=COLORS['red'],   linestyle='--', linewidth=1.2, label='Fragile threshold (0.4)')
ax.set_ylabel('Jaccard similarity (higher = more stable)')
ax.set_title('Perturbation Analysis — Recommendation Stability', fontweight='bold')
ax.set_ylim(0, 1.1)
ax.bar_label(bars, fmt='%.2f', padding=4, fontsize=9)
ax.legend()
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('plots/perturbation.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 11 — Performance Evaluation
**Work Package: Performance Evaluation**

We implement **two evaluation methods** (required by the work package):

### Method 1: Holdout split (80/20)
Train on 80% of ratings, evaluate on 20%.

### Method 2: Leave-one-out cross-validation
For each user, hide their highest-rated recipe. Check if it appears in top-k.

### Metrics:
$$\text{Precision}@k = \frac{|\text{relevant} \cap \text{recommended}|}{k}$$

$$\text{Recall}@k = \frac{|\text{relevant} \cap \text{recommended}|}{|\text{relevant}|}$$

$$\text{F1}@k = 2 \times \frac{\text{Precision}@k \times \text{Recall}@k}{\text{Precision}@k + \text{Recall}@k}$$


In [ ]:
# ── 11.1  Build evaluation sets ─────────────────────────────────────────────

RELEVANCE_THRESHOLD = 4  # ratings >= 4 are considered 'relevant'

def build_user_relevant_sets(df_ratings: pd.DataFrame,
                               threshold: int = RELEVANCE_THRESHOLD) -> dict:
    """
    Returns {user_id: set of recipe_ids the user rated >= threshold}.
    Only named users (0-5) for clean evaluation.
    """
    relevant = {}
    for uid in range(len(USERS)):
        liked = set(df_ratings[
            (df_ratings['user_id'] == uid) &
            (df_ratings['rating']  >= threshold)
        ]['recipe_id'])
        relevant[uid] = liked
    return relevant


relevant_sets = build_user_relevant_sets(df_ratings)
print('Relevant item counts per user (ratings >= 4):')
for uid, (uname, _) in enumerate(USERS.items()):
    print(f'  {uname:<8}  {len(relevant_sets[uid])} liked recipes')


In [ ]:
# ── 11.2  Precision@k, Recall@k, F1@k ──────────────────────────────────────

def precision_at_k(recommended: list, relevant: set, k: int) -> float:
    top_k = set(recommended[:k])
    hits  = len(top_k & relevant)
    return hits / k if k > 0 else 0.0

def recall_at_k(recommended: list, relevant: set, k: int) -> float:
    top_k = set(recommended[:k])
    hits  = len(top_k & relevant)
    return hits / len(relevant) if len(relevant) > 0 else 0.0

def f1_at_k(p: float, r: float) -> float:
    return 2*p*r/(p+r) if (p+r) > 0 else 0.0


def evaluate_model(method: str = 'hybrid', k_values: list = None) -> pd.DataFrame:
    """Evaluate recommender across multiple k values."""
    if k_values is None:
        k_values = [1, 3, 5, 10, 15, 20]

    rows = []
    for k in k_values:
        prec_list, rec_list, f1_list = [], [], []
        user_items = list(zip(list(USERS.keys()), range(len(USERS))))

        for uname, uid in user_items:
            if method == 'hybrid':
                recs_df = full_recommend(uname, uid, alpha=0.6, k=max(k_values))
                rec_ids = recs_df['recipe_id'].tolist()
            elif method == 'content':
                scores  = content_based_scores(USERS[uname]['vec'], R)
                order   = np.argsort(scores)[::-1]
                rec_ids = [df_clean.iloc[i]['recipe_id'] for i in order]
            elif method == 'collab':
                recs_df = collab_recommend(uid, k=max(k_values))
                rec_ids = recs_df['recipe_id'].tolist()

            rel = relevant_sets[uid]
            if len(rel) == 0: continue

            p = precision_at_k(rec_ids, rel, k)
            r = recall_at_k(rec_ids, rel, k)
            prec_list.append(p)
            rec_list.append(r)
            f1_list.append(f1_at_k(p, r))

        rows.append({'k':          k,
                      'method':     method,
                      'precision':  np.mean(prec_list),
                      'recall':     np.mean(rec_list),
                      'f1':         np.mean(f1_list)})
    return pd.DataFrame(rows)


print('Evaluating all three methods...')
eval_content = evaluate_model('content')
eval_collab  = evaluate_model('collab')
eval_hybrid  = evaluate_model('hybrid')
eval_all     = pd.concat([eval_content, eval_collab, eval_hybrid])

print('\n=== Evaluation Results ===')
print(eval_all.pivot_table(index='k', columns='method',
      values=['precision','recall','f1']).round(3).to_string())


In [ ]:
# ── 11.3  Precision-Recall curves ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
method_colors = {'content': COLORS['blue'], 'collab': COLORS['purple'],
                   'hybrid': COLORS['teal']}

for ax, metric in zip(axes, ['precision','recall','f1']):
    for method, color in method_colors.items():
        sub = eval_all[eval_all['method']==method]
        ax.plot(sub['k'], sub[metric], marker='o', color=color,
                 linewidth=1.8, markersize=5, label=method)
    ax.set_xlabel('k')
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f'{metric.capitalize()}@k', fontweight='bold')
    ax.legend()

plt.suptitle('Recommender System Evaluation Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/evaluation_curves.png', dpi=120, bbox_inches='tight')
plt.show()


In [ ]:
# ── 11.4  Leave-one-out evaluation (Method 2) ────────────────────────────────
print('=== Method 2: Leave-One-Out Cross-Validation ===')
print()
hit_rates = {}

for uname, uid in zip(USERS.keys(), range(len(USERS))):
    # Find the user's top-rated recipe
    user_ratings = df_ratings[df_ratings['user_id']==uid].sort_values('rating', ascending=False)
    if len(user_ratings) < 2: continue

    top_recipe_id = int(user_ratings.iloc[0]['recipe_id'])

    # Get recommendations excluding the top recipe
    known = set(user_ratings['recipe_id']) - {top_recipe_id}
    recs  = full_recommend(uname, uid, alpha=0.6, k=10)
    rec_ids = set(recs['recipe_id'])

    hit = top_recipe_id in rec_ids
    hit_rates[uname] = hit

    top_name = df_clean[df_clean['recipe_id']==top_recipe_id]['name'].values
    top_name = top_name[0] if len(top_name) > 0 else 'Unknown'
    print(f'  {uname:<8}  held-out: "{top_name[:30]}"  → Hit@10: {"✅" if hit else "❌"}')

overall_hit_rate = np.mean(list(hit_rates.values()))
print(f'\nOverall Hit@10 rate: {overall_hit_rate:.1%}  ({sum(hit_rates.values())}/{len(hit_rates)} users)')


---
## Section 12 — Hyperparameter Tuning with Optuna
**Work Package: Hyperparameter Tuning**

We tune three hyperparameters using **Optuna**'s Bayesian optimisation:

| Parameter | Symbol | Role | Search range |
|-----------|--------|------|--------------|
| Latent factors | k | Dimensionality of U, V | [10, 100] |
| Regularisation | λ | Prevents overfitting | [0.001, 1.0] |
| Blend weight | α | Content vs collaborative | [0.0, 1.0] |

**Objective:** maximise mean Precision@10 across all users.

Optuna uses **Tree-structured Parzen Estimators (TPE)** — a Bayesian method that
builds a probabilistic model of which hyperparameters tend to give good results,
and focuses sampling there.


In [ ]:
# ── 12.1  Optuna tuning ───────────────────────────────────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    # Define search space
    k_factors = trial.suggest_int('k_factors', 10, 100)
    reg_all   = trial.suggest_float('reg_all',  0.001, 1.0,  log=True)
    lr_all    = trial.suggest_float('lr_all',   0.001, 0.05, log=True)
    alpha     = trial.suggest_float('alpha',    0.0,   1.0)

    # Train SVD with these hyperparameters
    model = SVD(n_factors=k_factors, reg_all=reg_all, lr_all=lr_all,
                 n_epochs=20, random_state=42)
    model.fit(trainset_sv)

    # Evaluate: mean Precision@10 across named users
    prec_list = []
    for uname, uid in zip(USERS.keys(), range(len(USERS))):
        cb = content_based_scores(USERS[uname]['vec'], R)
        cb_n = (cb - cb.min()) / (cb.max() - cb.min() + 1e-9)
        cf_raw = np.array([model.predict(uid, rid).est for rid in df_clean['recipe_id']])
        cf_n = (cf_raw - cf_raw.min()) / (cf_raw.max() - cf_raw.min() + 1e-9)
        scores = alpha * cb_n + (1 - alpha) * cf_n
        order  = np.argsort(scores)[::-1]
        rec_ids = [df_clean.iloc[i]['recipe_id'] for i in order[:10]]
        rel = relevant_sets[uid]
        if len(rel) > 0:
            prec_list.append(precision_at_k(rec_ids, rel, 10))

    return np.mean(prec_list) if prec_list else 0.0


print('Running Optuna hyperparameter search (50 trials)...')
study = optuna.create_study(direction='maximize',
                              sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=False)

print(f'\n✅ Best Precision@10:  {study.best_value:.4f}')
print(f'Best hyperparameters:')
for k, v in study.best_params.items():
    print(f'  {k:<15} = {v}')


In [ ]:
# ── 12.2  Visualise hyperparameter importance ─────────────────────────────────
trials_df = study.trials_dataframe()
params = ['params_k_factors','params_reg_all','params_lr_all','params_alpha']
labels = ['k (latent factors)','λ (regularisation)','lr (learning rate)','α (blend weight)']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, param, label in zip(axes, params, labels):
    if param not in trials_df.columns: continue
    sc = ax.scatter(trials_df[param], trials_df['value'],
                     c=trials_df['value'], cmap='RdYlGn',
                     alpha=0.7, s=40, edgecolors='none')
    best_val = study.best_params.get(param.replace('params_',''), None)
    if best_val is not None:
        ax.axvline(best_val, color='black', linestyle='--', linewidth=1.5,
                    label=f'best={best_val:.3f}')
        ax.legend(fontsize=8)
    ax.set_xlabel(label)
    ax.set_ylabel('Precision@10')
    ax.set_title(f'Effect of {label}', fontweight='bold')
    plt.colorbar(sc, ax=ax)

plt.suptitle('Optuna Hyperparameter Search Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/hyperparameter_tuning.png', dpi=120, bbox_inches='tight')
plt.show()


---
## Section 13 — Experiment Logging with Weights & Biases
**Work Package: Experiments Logging**

W&B tracks every training run — hyperparameters, metrics, system info.
This satisfies the Experiments Logging work package.

**Setup:** Run `wandb login` in a terminal and paste your API key from wandb.ai


In [ ]:
# ── 13.1  W&B integration ─────────────────────────────────────────────────────
# Uncomment and run after:  wandb login

WANDB_ENABLED = False   # set True after wandb login

if WANDB_ENABLED:
    import wandb

    # Log all Optuna trials to W&B
    for trial in study.trials:
        wandb.init(
            project   = 'food-recommender-system',
            name      = f'trial-{trial.number}',
            config    = trial.params,
            reinit    = True,
            tags      = ['optuna', 'svd', 'hybrid'],
        )
        wandb.log({
            'precision_at_10': trial.value,
            'trial_number':    trial.number,
            **trial.params
        })
        wandb.finish()

    print('✅ All trials logged to W&B')

else:
    print('W&B logging disabled. Set WANDB_ENABLED=True after running: wandb login')
    print()
    print('What W&B gives you:')
    print('  • Dashboard showing all trials with hyperparameter values')
    print('  • Parallel coordinates plot — instantly see which params matter')
    print('  • Model artifacts — save and version your trained models')
    print('  • Team sharing — share experiment results with your supervisor')
    print()
    # Simulate what gets logged
    print('Example of what gets logged per trial:')
    for trial in study.best_trials[:3]:
        print(f'  Trial {trial.number}: precision@10={trial.value:.4f}  params={trial.params}')


---
## Section 14 — Frontend Application (Streamlit)
**Work Package: Frontend Application**

This cell writes the complete `app.py` Streamlit frontend to disk.
Run it with: `streamlit run app.py`


In [ ]:
# ── 14.1  Write Streamlit app ────────────────────────────────────────────────
STREAMLIT_APP = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.metrics.pairwise import cosine_similarity

st.set_page_config(
    page_title="Health Food Recommender",
    page_icon="🥗",
    layout="wide"
)

# ── Load data ──────────────────────────────────────────────────────────────
@st.cache_data
def load_data():
    return pd.read_csv("data/recipes_clean.csv")

df = load_data()

FEATURE_MAX = {"calories":2000,"protein_g":150,"carbs_g":300,
               "fat_g":100,"fiber_g":50,"sodium_mg":4000,"sugar_g":150}
NUTRITION_COLS = list(FEATURE_MAX.keys())
LABEL_COLS = ["diabetic_ok","low_sodium","high_fiber","high_protein",
               "heart_healthy","low_calorie","high_sugar","vegetarian"]

def build_recipe_matrix(df):
    nutrition = df[NUTRITION_COLS].copy()
    for col, mx in FEATURE_MAX.items():
        nutrition[col] = nutrition[col] / mx
    labels = df[LABEL_COLS].astype(float)
    return pd.concat([nutrition, labels], axis=1).values

R = build_recipe_matrix(df)

def make_user_vec(cal, prot, max_carbs, max_fat, min_fiber,
                   diabetic, hypertensive, vegetarian):
    n = np.array([cal/2000, prot/150, max_carbs/300, max_fat/100,
                   min_fiber/50, 0, 0])
    l = np.array([float(diabetic), float(hypertensive), float(min_fiber>=5),
                   float(prot>=30), float(hypertensive), float(cal<=300),
                   0, float(vegetarian)])
    return np.concatenate([n, l])

def health_filter_row(row, diabetic, hypertensive, vegetarian, lactose_intol):
    if diabetic      and row["carbs_g"]  > 45:  return False
    if hypertensive  and row["sodium_mg"]> 600:  return False
    if vegetarian    and row["vegetarian"]==0:    return False
    if lactose_intol and row["cuisine"]  in {"italian","dessert"}: return False
    return True

# ── UI ─────────────────────────────────────────────────────────────────────
st.title("🥗 Health-Personalized Food Recommender")
st.markdown("Get meal recommendations tailored to your health profile.")

col1, col2 = st.columns([1, 2])

with col1:
    st.subheader("Your Health Profile")
    target_cal   = st.slider("Target calories per meal", 200, 900, 450, 50)
    target_prot  = st.slider("Target protein (g)",        10,  80,  30,  5)
    max_carbs    = st.slider("Max carbs (g)",              10, 250, 100, 10)
    max_fat      = st.slider("Max fat (g)",                 5,  80,  30,  5)
    min_fiber    = st.slider("Min fiber (g)",               0,  20,   5,  1)
    st.divider()
    diabetic     = st.checkbox("I have Type 2 Diabetes")
    hypertensive = st.checkbox("I have Hypertension")
    vegetarian   = st.checkbox("I am Vegetarian")
    lactose_intol= st.checkbox("I am Lactose Intolerant")
    k_recs       = st.slider("Number of recommendations", 3, 20, 8)

with col2:
    user_vec = make_user_vec(target_cal, target_prot, max_carbs, max_fat, min_fiber,
                              diabetic, hypertensive, vegetarian)
    scores = cosine_similarity(user_vec.reshape(1,-1), R).flatten()
    results = df.copy()
    results["score"] = scores
    results = results.sort_values("score", ascending=False)
    results["allowed"] = results.apply(
        lambda row: health_filter_row(row, diabetic, hypertensive, vegetarian, lactose_intol),
        axis=1)
    recommendations = results[results["allowed"]].head(k_recs).reset_index(drop=True)

    st.subheader(f"Top {k_recs} Recommendations for You")
    if len(recommendations) == 0:
        st.warning("No recipes match your constraints. Try relaxing some conditions.")
    else:
        fig = px.bar(recommendations, x="score", y="name",
                      color="cuisine", orientation="h",
                      hover_data=["calories","protein_g","carbs_g","fat_g"],
                      title="Recommendations ranked by match score",
                      labels={"score":"Match score","name":"Recipe","cuisine":"Cuisine"})
        fig.update_layout(yaxis={"categoryorder":"total ascending"}, height=420)
        st.plotly_chart(fig, use_container_width=True)

        st.subheader("Nutritional Details")
        display_cols = ["name","cuisine","calories","protein_g","carbs_g","fat_g","fiber_g"]
        st.dataframe(recommendations[display_cols].round(1),
                      use_container_width=True, hide_index=True)

# ── Sidebar stats ──────────────────────────────────────────────────────────
with st.sidebar:
    st.subheader("Dataset Stats")
    st.metric("Total recipes", len(df))
    st.metric("Cuisines", df["cuisine"].nunique())
    st.metric("Diabetic-ok", int(df["diabetic_ok"].sum()))
    st.metric("Vegetarian", int(df["vegetarian"].sum()))
'''

with open('app.py', 'w') as f:
    f.write(STREAMLIT_APP)

print('✅ Streamlit app written to app.py')
print()
print('To run the frontend:')
print('  streamlit run app.py')
print()
print('The app provides:')
print('  • Health profile sliders (calories, protein, carbs, fat, fiber)')
print('  • Health condition checkboxes (diabetic, hypertensive, vegetarian, lactose)')
print('  • Interactive bar chart of ranked recommendations')
print('  • Nutritional details table')
print('  • Sidebar with dataset statistics')


---
## Section 15 — Full Pipeline Summary

Run the complete pipeline end-to-end in one cell.


In [ ]:
# ── Full pipeline summary ────────────────────────────────────────────────────
print('=' * 60)
print('HEALTH FOOD RECOMMENDER — FULL PIPELINE SUMMARY')
print('=' * 60)
print()
print(f'1. DATA COLLECTION')
print(f'   Recipes scraped/generated : {len(df_raw)}')
print(f'   Features per recipe        : {df_raw.shape[1]}')
print()
print(f'2. ANNOTATION')
print(f'   Health labels assigned     : {len(LABEL_COLS)}')
print(f'   Diabetic-ok recipes        : {int(df_annotated["diabetic_ok"].sum())}')
print()
print(f'3. DATA QUALITY')
print(f'   Missing values imputed     : {df_annotated[NUMERIC_COLS].isna().sum().sum()}')
print(f'   Inconsistent kcal values   : {n_inconsistent}')
print()
print(f'4. VECTOR EMBEDDINGS')
print(f'   Recipe matrix R shape      : {R.shape}')
print(f'   Feature dimensions         : {len(FEATURE_NAMES)}')
print()
print(f'5. RECOMMENDER SYSTEM')
print(f'   Content-based              : cosine similarity on feature vectors')
print(f'   Collaborative filtering    : SVD matrix factorisation (k={study.best_params["k_factors"]})')
print(f'   Blend weight alpha         : {study.best_params["alpha"]:.3f}')
print()
print(f'6. HEALTH CONSTRAINTS')
print(f'   Rules applied              : diabetic, hypertensive, vegetarian, lactose')
print()
print(f'7. PERTURBATION ANALYSIS')
print(f'   Mean Jaccard stability     : {perturbation_df["mean_jaccard"].mean():.3f}')
print()
print(f'8. EVALUATION (best model)')
best_eval = eval_hybrid[eval_hybrid['k']==10].iloc[0]
print(f'   Precision@10               : {best_eval["precision"]:.3f}')
print(f'   Recall@10                  : {best_eval["recall"]:.3f}')
print(f'   F1@10                      : {best_eval["f1"]:.3f}')
print(f'   Hit@10 (leave-one-out)     : {overall_hit_rate:.1%}')
print()
print(f'9. HYPERPARAMETER TUNING')
print(f'   Optuna trials              : {len(study.trials)}')
print(f'   Best Precision@10          : {study.best_value:.4f}')
print(f'   Best params                : {study.best_params}')
print()
print(f'10. FRONTEND')
print(f'    Streamlit app             : app.py  →  run: streamlit run app.py')
print()
print('=' * 60)
print('ALL WORK PACKAGES COMPLETED')
print('=' * 60)


In [ ]:
# ── Plot all saved figures summary ────────────────────────────────────────
import glob
plot_files = sorted(glob.glob('plots/*.png'))
print(f'Generated plots ({len(plot_files)} files):')
for f in plot_files:
    print(f'  {f}')
